In [ ]:
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from scipy.stats import norm

try:
    import lpips
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    lpips_model = lpips.LPIPS(net='alex').to(device)
    lpips_model.eval()
    LPIPS_AVAILABLE = True
except ImportError:
    LPIPS_AVAILABLE = False
    print("lpips not installed -- LPIPS column will be skipped.")

# ==========================================
# CONFIG
# ==========================================
METHOD_FOLDERS = {
    "GAN":                          Path("pix2pix_baseline"),
    "TransUNet":                    Path("transformer_full_val_eval"),
    "LDM baseline (linear)":        Path("evaluation_results_Linear/generated_samples"),
    "LDM baseline (scaled_linear)": Path("evaluation_results_scaled_basic/generated_samples"),
    "LDM final (scaled_linear)":    Path("evaluation_results_THESIS_FINAL_2/generated_samples"),
}

# ==========================================
# STEP 1: common keys across all methods (unchanged)
# ==========================================
def get_available_keys(folder):
    return {f.stem for f in folder.glob("*.pt")}

key_sets = {name: get_available_keys(folder) for name, folder in METHOD_FOLDERS.items()}
common_keys = set.intersection(*key_sets.values())

print("Per-method file counts:")
for name, keys in key_sets.items():
    print(f"  {name}: {len(keys)} files")
print(f"\nCommon (case, slice) keys across ALL methods: {len(common_keys)}")

if len(common_keys) == 0:
    raise RuntimeError("No overlapping samples found across all methods -- check paths/filenames.")

common_keys = sorted(common_keys)

# ==========================================
# STEP 2: shared metric functions
# ==========================================
def compute_all_metrics(real, synth):
    if real.shape != synth.shape:
        from skimage.transform import resize
        synth = resize(synth, real.shape)

    data_range = max(np.max(real) - np.min(real), 1e-6)

    try:
        p = psnr(real, synth, data_range=data_range)
    except Exception:
        p = np.nan
    try:
        win_size = min(7, min(real.shape) // 2)
        if win_size % 2 == 0:
            win_size -= 1
        s = ssim(real, synth, data_range=data_range, win_size=max(3, win_size))
    except Exception:
        s = np.nan

    mae = np.mean(np.abs(real - synth))

    l = np.nan
    if LPIPS_AVAILABLE:
        real_t = torch.from_numpy(real * 2 - 1).float().unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1).to(device)
        synth_t = torch.from_numpy(synth * 2 - 1).float().unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1).to(device)
        with torch.no_grad():
            l = lpips_model(real_t, synth_t).item()

    return p, s, l, mae

def calculate_clinical_metrics(real_img, gen_img, mask_img):
    """gCNR/d' computed on the GENERATED image, using the established (corrected) formula:
    gCNR uses z_score (un-pooled denominator); d'/gSNR uses the pooled-noise denominator.
    These are two DIFFERENT quantities -- do not substitute one for the other."""
    fg_mask = mask_img > 0
    bg_mask = mask_img == 0
    if np.sum(fg_mask) == 0 or np.sum(bg_mask) == 0:
        return np.nan, np.nan

    gen_fg = gen_img[fg_mask].flatten()
    gen_bg = gen_img[bg_mask].flatten()
    mu_fg, mu_bg = np.mean(gen_fg), np.mean(gen_bg)
    std_fg, std_bg = np.std(gen_fg), np.std(gen_bg)

    pooled_noise = np.sqrt(0.5 * (std_fg**2 + std_bg**2))
    if pooled_noise == 0:
        return np.nan, np.nan
    signal_diff = np.abs(mu_fg - mu_bg)

    d_prime = signal_diff / pooled_noise  # gSNR
    z_score = signal_diff / np.sqrt(std_fg**2 + std_bg**2)
    gcnr = 1 - 2 * norm.cdf(-z_score / 2)

    return gcnr, d_prime

# ==========================================
# STEP 3: run every method on the exact same common_keys
# ==========================================
results = {name: {'psnr': [], 'ssim': [], 'lpips': [], 'mae': [], 'gcnr': [], 'd_prime': []}
           for name in METHOD_FOLDERS}

missing_mask_warned = set()

for key in common_keys:
    for name, folder in METHOD_FOLDERS.items():
        d = torch.load(folder / f"{key}.pt", weights_only=False)
        real = np.asarray(d['real_flair'])
        synth = np.asarray(d['synthetic_flair'])

        p, s, l, mae = compute_all_metrics(real, synth)
        results[name]['psnr'].append(p)
        results[name]['ssim'].append(s)
        results[name]['lpips'].append(l)
        results[name]['mae'].append(mae)

        mask = d.get('mask', None)
        if mask is None:
            if name not in missing_mask_warned:
                print(f"WARNING: '{name}' has no 'mask' key -- gCNR/d' will be NaN for all its samples.")
                missing_mask_warned.add(name)
            gcnr, d_prime = np.nan, np.nan
        else:
            mask = np.asarray(mask)
            gcnr, d_prime = calculate_clinical_metrics(real, synth, mask)

        results[name]['gcnr'].append(gcnr)
        results[name]['d_prime'].append(d_prime)

# ==========================================
# STEP 4: ground-truth reference gCNR/d' (using real_flair as both "real" and "gen" is meaningless --
# instead compute GT detectability directly from real_flair vs itself's own fg/bg stats)
# ==========================================
def ground_truth_clinical_metrics(real_img, mask_img):
    fg_mask = mask_img > 0
    bg_mask = mask_img == 0
    if np.sum(fg_mask) == 0 or np.sum(bg_mask) == 0:
        return np.nan, np.nan
    fg = real_img[fg_mask].flatten()
    bg = real_img[bg_mask].flatten()
    mu_fg, mu_bg = np.mean(fg), np.mean(bg)
    std_fg, std_bg = np.std(fg), np.std(bg)
    pooled_noise = np.sqrt(0.5 * (std_fg**2 + std_bg**2))
    if pooled_noise == 0:
        return np.nan, np.nan
    signal_diff = np.abs(mu_fg - mu_bg)
    d_prime = signal_diff / pooled_noise
    z_score = signal_diff / np.sqrt(std_fg**2 + std_bg**2)
    gcnr = 1 - 2 * norm.cdf(-z_score / 2)
    return gcnr, d_prime

gt_gcnr_list, gt_dprime_list = [], []
any_folder = next(iter(METHOD_FOLDERS.values()))
for key in common_keys:
    d = torch.load(any_folder / f"{key}.pt", weights_only=False)
    mask = d.get('mask', None)
    if mask is not None:
        gcnr, dprime = ground_truth_clinical_metrics(np.asarray(d['real_flair']), np.asarray(mask))
        gt_gcnr_list.append(gcnr)
        gt_dprime_list.append(dprime)

gt_gcnr_arr = np.array(gt_gcnr_list, dtype=float)
gt_dprime_arr = np.array(gt_dprime_list, dtype=float)
gt_gcnr_mean = np.nanmean(gt_gcnr_arr)
gt_dprime_mean = np.nanmean(gt_dprime_arr)

# ==========================================
# STEP 5: build the summary table
# ==========================================
summary_rows = []
for name in METHOD_FOLDERS:
    row = {"Method": name}
    for metric_key, label in [('psnr', 'PSNR'), ('ssim', 'SSIM'), ('lpips', 'LPIPS'), ('mae', 'MAE')]:
        vals = np.array(results[name][metric_key], dtype=float)
        vals = vals[~np.isnan(vals)]
        row[label] = f"{np.mean(vals):.4f} ± {np.std(vals):.4f}" if len(vals) > 0 else "N/A"

    gcnr_vals = np.array(results[name]['gcnr'], dtype=float)
    gcnr_vals = gcnr_vals[~np.isnan(gcnr_vals)]
    dprime_vals = np.array(results[name]['d_prime'], dtype=float)
    dprime_vals = dprime_vals[~np.isnan(dprime_vals)]

    if len(gcnr_vals) > 0:
        gcnr_mean = np.mean(gcnr_vals)
        row['gCNR'] = f"{gcnr_mean:.4f} ± {np.std(gcnr_vals):.4f}"
        row['gCNR retention'] = f"{100 * gcnr_mean / gt_gcnr_mean:.1f}%"
    else:
        row['gCNR'], row['gCNR retention'] = "N/A", "N/A"

    if len(dprime_vals) > 0:
        dprime_mean = np.mean(dprime_vals)
        row["d'/gSNR"] = f"{dprime_mean:.4f} ± {np.std(dprime_vals):.4f}"
        row["gSNR retention"] = f"{100 * dprime_mean / gt_dprime_mean:.1f}%"
    else:
        row["d'/gSNR"], row["gSNR retention"] = "N/A", "N/A"

    row["n (all)"] = len(common_keys)
    row["n (tumor slices)"] = len(gcnr_vals)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 100)
print(f"FAIR COMPARISON TABLE (n={len(common_keys)} total, ground-truth gCNR={gt_gcnr_mean:.4f}, gSNR={gt_dprime_mean:.4f})")
print("=" * 100)
print(summary_df.to_string(index=False))



In [ ]:
from generative.networks.nets import AutoencoderKL, DiffusionModelUNet
from generative.networks.schedulers import DDIMScheduler
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vae = AutoencoderKL(
    spatial_dims=2, in_channels=1, out_channels=1,
    num_channels=(64, 128, 256), latent_channels=8,
    num_res_blocks=2, attention_levels=(False, False, True),
    norm_num_groups=32,
).to(device)
vae_ckpt = torch.load("results/vae_mixed/autoencoder_best.pth", map_location=device, weights_only=False)
vae.load_state_dict(vae_ckpt["model"])          # <-- was missing
vae.eval()                                       # <-- was missing

unet = DiffusionModelUNet(
    spatial_dims=2, in_channels=16, out_channels=8,
    num_channels=(64, 128, 256), attention_levels=(False, True, True),
    num_head_channels=(32, 64, 128), num_res_blocks=3,
).to(device)
unet_ckpt = torch.load("diffusion/diffusion_best2.pth", map_location=device, weights_only=False)
unet_state = unet.state_dict()
for name, value in unet_ckpt["ema_state_dict"].items():   # <-- was missing entirely
    if name in unet_state:
        unet_state[name] = value
unet.load_state_dict(unet_state)
unet.eval()                                      # <-- was missing

ddim_scheduler = DDIMScheduler(
    num_train_timesteps=1000,
    schedule="scaled_linear_beta",
    beta_start=0.00085,
    beta_end=0.012,
    clip_sample=False,
    set_alpha_to_one=False,
)

latent_stats = torch.load("latent_pairs/latent_stats.pt", weights_only=False)
LATENT_MEAN = latent_stats["mean"].to(device).view(1, -1, 1, 1)
LATENT_STD = latent_stats["std"].to(device).view(1, -1, 1, 1)

In [ ]:
import numpy as np
import pandas as pd
import torch
import nibabel as nib
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from pathlib import Path
from scipy.stats import ttest_rel
from tqdm.auto import tqdm

from src.latent_dataset import PairedLatentDataset


# ==================================================================
# CONFIG
# ==================================================================

DATA_ROOT = Path(
    "BraTS2020_TrainingData/"
    "BraTS2020_TrainingData/"
    "MICCAI_BraTS2020_TrainingData"
)

VAL_PAIRS = "latent_pairs/val_pairs_scaled.pt"

# Generated-sample locations (per case/slice key)
LDM_FINAL_DIR = "evaluation_results_scaled_tuned/generated_samples"
LDM_BASE_DIR = "evaluation_results_Linear/generated_samples"
GAN_DIR = "pix2pix_baseline"
TRANS_DIR = "transformer_full_val_eval"

# Outputs
METRICS_CSV = "fair_comparison_raw_values.csv"
COND_CSV_PATH = "conditioning_test_improved.csv"
GALLERY_PATH = "figure2_main_gallery.png"
COND_FIG_PATH = "figure3_conditioning_comparison.png"

# Sampling settings
GUIDANCE = 2.6
ETA = 0.0                  # deterministic DDIM
DDIM_STEPS = 200
NUM_COND_PAIRS = 25
MIN_TUMOR_SIZE = 50        # px required for conditioning-test eligibility
NOISE_SEED_BASE = 1234
RNG_SEED = 42

# Visualization settings
ERROR_VMIN = 0.0           # shared color domain for ALL error/difference maps
ERROR_VMAX = 0.3           # (Figure 2 error maps == Figure 3 difference maps)
BOX_TARGET = "red"         # solid: target patient's tumor
BOX_WRONG = "orange"       # dashed: wrong-condition patient's tumor
BOX_ON_ERROR = "cyan"
BOX_MIN_SPAN = 10          # px, keeps small-tumor boxes visible
FIG_DPI = 200


# ==================================================================
# SEGMENTATION MASKS + TUMOR STATISTICS
# ==================================================================

def build_seg_lookup(val_dataset, data_root):
    """
    dataset index -> 2D tumor mask cropped to the same 160x160 region
    used by the latent dataset. Seg volumes are cached per case.
    """
    lookup, seg_cache = {}, {}

    for i in tqdm(range(len(val_dataset)),
                  desc="Building segmentation lookup", unit="slice"):
        sample = val_dataset[i]
        case, slice_idx = sample["case"], sample["slice_idx"]

        if case not in seg_cache:
            seg_file = data_root / case / f"{case}_seg.nii"
            seg_cache[case] = (
                nib.load(seg_file).get_fdata() if seg_file.exists() else None
            )

        seg_vol = seg_cache[case]
        if seg_vol is not None and slice_idx < seg_vol.shape[2]:
            lookup[i] = seg_vol[48:208, 48:208, slice_idx]
        else:
            lookup[i] = np.zeros((160, 160), dtype=np.float32)

    return lookup


def tumor_centroid_and_size(mask):
    """-> (centroid [y, x], tumor pixel count). Falls back to image center."""
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return np.array([80.0, 80.0], dtype=np.float32), 0
    return np.array([ys.mean(), xs.mean()], dtype=np.float32), len(ys)


def precompute_tumor_stats(seg_lookup):
    """index -> (centroid, size), computed once and reused everywhere."""
    return {i: tumor_centroid_and_size(m) for i, m in seg_lookup.items()}


def find_most_different_tumor(target_idx, tumor_stats, min_size=MIN_TUMOR_SIZE):
    """
    Index of the tumor that differs most from target_idx in
    centroid location + size, or None if no candidate exists.
    """
    t_centroid, t_size = tumor_stats[target_idx]
    if t_size == 0:
        return None

    best_j, best_dist = None, -1.0
    for j, (centroid, size) in tumor_stats.items():
        if j == target_idx or size < min_size:
            continue
        dist = np.linalg.norm(t_centroid - centroid) + abs(t_size - size) / 100.0
        if dist > best_dist:
            best_dist, best_j = dist, j
    return best_j


# ==================================================================
# DRAWING HELPERS
# ==================================================================

def draw_tumor_box(ax, mask, color=BOX_TARGET, linewidth=2.0,
                   linestyle="-", min_span=BOX_MIN_SPAN):
    """
    Bounding box around the tumor.

    Boxes are padded up to at least `min_span` pixels so that tiny
    tumors (e.g. the "Small tumor" row of the gallery) stay visible.
    """
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return

    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()

    if (y1 - y0) < min_span:
        pad = (min_span - (y1 - y0)) / 2
        y0, y1 = y0 - pad, y1 + pad
    if (x1 - x0) < min_span:
        pad = (min_span - (x1 - x0)) / 2
        x0, x1 = x0 - pad, x1 + pad

    ax.add_patch(
        Rectangle((x0, y0), x1 - x0, y1 - y0,
                  linewidth=linewidth, linestyle=linestyle,
                  edgecolor=color, facecolor="none")
    )


def show_error_map(ax, err, mask=None, title=None):
    """
    Show an error/difference map on the SHARED color domain
    (ERROR_VMIN..ERROR_VMAX) so maps are comparable across figures.
    """
    im = ax.imshow(err, cmap="hot", vmin=ERROR_VMIN, vmax=ERROR_VMAX)
    if mask is not None:
        draw_tumor_box(ax, mask, color=BOX_ON_ERROR)
    if title:
        ax.set_title(title, fontsize=10)
    ax.axis("off")
    return im


# ==================================================================
# DIFFUSION SAMPLING
# ==================================================================

@torch.no_grad()
def sample_with_condition(unet, scheduler, condition, start_latent,
                          guidance=GUIDANCE, eta=ETA, steps=DDIM_STEPS,
                          progress_desc="DDIM sampling"):
    """Classifier-free-guided DDIM sampling (deterministic when eta=0)."""
    scheduler.set_timesteps(steps)
    latent = start_latent.clone()

    for t in tqdm(scheduler.timesteps, desc=progress_desc, unit="step", leave=False):
        timestep = torch.full((latent.shape[0],), int(t),
                              device=latent.device, dtype=torch.long)

        noise_cond = unet(torch.cat([latent, condition], dim=1), timestep)
        noise_uncond = unet(
            torch.cat([latent, torch.zeros_like(condition)], dim=1), timestep
        )
        noise_pred = noise_uncond + guidance * (noise_cond - noise_uncond)

        latent, _ = scheduler.step(noise_pred, int(t), latent, eta=eta)

    return latent


def to_img(z, LATENT_MEAN, LATENT_STD, vae):
    """Normalized latent -> [0, 1] numpy image."""
    img = vae.decode(z * LATENT_STD + LATENT_MEAN)
    img = (torch.clamp(img, -1, 1) + 1) / 2
    return img.squeeze().detach().cpu().numpy()


def sample_correct_wrong_pair(target_idx, wrong_idx, val_dataset,
                              unet, scheduler, vae,
                              LATENT_MEAN, LATENT_STD, device,
                              guidance=GUIDANCE, eta=ETA, steps=DDIM_STEPS):
    """
    Sample twice from the SAME starting noise:
      - once with the correct T1ce condition
      - once with a deliberately mismatched condition
    ...then decode everything needed for metrics and figures.
    """
    sample = val_dataset[target_idx]
    cond_correct = sample["condition"].unsqueeze(0).to(device)
    target = sample["target"].unsqueeze(0).to(device)
    cond_wrong = val_dataset[wrong_idx]["condition"].unsqueeze(0).to(device)

    torch.manual_seed(NOISE_SEED_BASE + int(target_idx))
    start_noise = torch.randn_like(target)

    z_correct = sample_with_condition(unet, scheduler, cond_correct, start_noise,
                                      guidance, eta, steps, "  DDIM correct")
    z_wrong = sample_with_condition(unet, scheduler, cond_wrong, start_noise,
                                    guidance, eta, steps, "  DDIM wrong")

    return {
        "t1ce_correct": to_img(cond_correct, LATENT_MEAN, LATENT_STD, vae),
        "t1ce_wrong": to_img(cond_wrong, LATENT_MEAN, LATENT_STD, vae),
        "real_flair": to_img(target, LATENT_MEAN, LATENT_STD, vae),
        "out_correct": to_img(z_correct, LATENT_MEAN, LATENT_STD, vae),
        "out_wrong": to_img(z_wrong, LATENT_MEAN, LATENT_STD, vae),
    }


def region_split_error(real_img, synth_img, mask):
    """-> (whole-image MAE, tumor-region MAE). Tumor MAE is NaN if no tumor."""
    whole_mae = np.mean(np.abs(real_img - synth_img))
    if np.sum(mask > 0) > 0:
        tumor_mae = np.mean(np.abs(real_img[mask > 0] - synth_img[mask > 0]))
    else:
        tumor_mae = np.nan
    return whole_mae, tumor_mae


# ==================================================================
# PART A: CONDITIONING SENSITIVITY TEST
# ==================================================================

def run_improved_conditioning_test(unet, scheduler, val_dataset, seg_lookup,
                                   LATENT_MEAN, LATENT_STD, vae, device,
                                   num_pairs=NUM_COND_PAIRS, eta=ETA,
                                   guidance=GUIDANCE, steps=DDIM_STEPS,
                                   min_tumor_size=MIN_TUMOR_SIZE):
    """
    For each selected sample: generate with the correct condition and with
    a deliberately mismatched condition (same starting noise), then compare
    reconstruction error inside vs. outside the tumor region.
    """
    print("\n" + "-" * 80)
    print("PREPARING CONDITIONING SENSITIVITY TEST")
    print("-" * 80)

    tumor_stats = precompute_tumor_stats(seg_lookup)

    valid_indices = [
        i for i, (_, size) in tumor_stats.items() if size >= min_tumor_size
    ]
    print(f"Valid samples with tumor >= {min_tumor_size} px: {len(valid_indices)}")
    if not valid_indices:
        raise RuntimeError("No valid samples found for conditioning test.")

    rng = np.random.RandomState(RNG_SEED)
    test_indices = rng.choice(
        valid_indices, size=min(num_pairs, len(valid_indices)), replace=False
    )
    print(f"Testing {len(test_indices)} pairs...")

    rows = []
    for pair_no, i in enumerate(
        tqdm(test_indices, desc="Conditioning sensitivity test", unit="pair"), 1
    ):
        j = find_most_different_tumor(i, tumor_stats, min_size=min_tumor_size)
        if j is None:
            print(f"  Pair {pair_no}: no mismatched tumor for idx {i}, skipping.")
            continue

        imgs = sample_correct_wrong_pair(
            i, j, val_dataset, unet, scheduler, vae,
            LATENT_MEAN, LATENT_STD, device,
            guidance=guidance, eta=eta, steps=steps,
        )

        mask = seg_lookup[i]
        real = imgs["real_flair"]
        whole_c, tumor_c = region_split_error(real, imgs["out_correct"], mask)
        whole_w, tumor_w = region_split_error(real, imgs["out_wrong"], mask)

        t_centroid, t_size = tumor_stats[i]
        w_centroid, w_size = tumor_stats[j]

        rows.append({
            "idx": int(i),
            "wrong_idx": int(j),
            "mse_correct": float(np.mean((imgs["out_correct"] - real) ** 2)),
            "mse_wrong": float(np.mean((imgs["out_wrong"] - real) ** 2)),
            "whole_mae_correct": whole_c,
            "whole_mae_wrong": whole_w,
            "tumor_mae_correct": tumor_c,
            "tumor_mae_wrong": tumor_w,
            "tumor_centroid_distance_px": float(np.linalg.norm(t_centroid - w_centroid)),
            "tumor_size_diff_px": float(abs(t_size - w_size)),
        })

        print(
            f"  Pair {pair_no}/{len(test_indices)}: idx {i} vs {j} | "
            f"whole MAE {whole_c:.4f} (correct) vs {whole_w:.4f} (wrong) | "
            f"tumor MAE {tumor_c:.4f} vs {tumor_w:.4f}"
        )

    df = pd.DataFrame(rows)
    df.to_csv(COND_CSV_PATH, index=False)
    print(f"\nResults saved to: {COND_CSV_PATH}")

    _print_conditioning_summary(df, eta)
    return df


def _print_conditioning_summary(df, eta):
    print("\n" + "=" * 80)
    print(f"IMPROVED CONDITIONING TEST (n={len(df)}, eta={eta}, mismatched condition)")
    print("=" * 80)

    if len(df) == 0:
        print("No valid test pairs were produced.")
        return

    metric_cols = [
        "mse_correct", "mse_wrong",
        "whole_mae_correct", "whole_mae_wrong",
        "tumor_mae_correct", "tumor_mae_wrong",
        "tumor_centroid_distance_px", "tumor_size_diff_px",
    ]
    print(df[metric_cols].describe())

    tumor = df.dropna(subset=["tumor_mae_correct", "tumor_mae_wrong"])
    if len(tumor) >= 2:
        t, p = ttest_rel(tumor["tumor_mae_correct"], tumor["tumor_mae_wrong"])
        print(f"\nPaired t-test, TUMOR REGION MAE (correct vs wrong): "
              f"t={t:.3f}, p={p:.4f}")
    else:
        print("\nNot enough valid tumor-region pairs for tumor MAE t-test.")

    t, p = ttest_rel(df["whole_mae_correct"], df["whole_mae_wrong"])
    print(f"\nPaired t-test, WHOLE IMAGE MAE (correct vs wrong): "
          f"t={t:.3f}, p={p:.4f}")
    print(f"\nMean tumor centroid distance (correct vs wrong pair): "
          f"{df['tumor_centroid_distance_px'].mean():.1f} px")
    print(f"Mean tumor size difference: "
          f"{df['tumor_size_diff_px'].mean():.1f} px")


# ==================================================================
# PART B: LOAD ALL METHODS FOR ONE CASE
# ==================================================================

def _load_generated(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing generated sample: {path}")
    return torch.load(path, weights_only=False)


def load_all_methods(key):
    """-> (ldm_final, ldm_base, gan, trans) dicts for one case/slice key."""
    return (
        _load_generated(f"{LDM_FINAL_DIR}/{key}.pt"),
        _load_generated(f"{LDM_BASE_DIR}/{key}.pt"),
        _load_generated(f"{GAN_DIR}/{key}.pt"),
        _load_generated(f"{TRANS_DIR}/{key}.pt"),
    )


def select_four_cases(metrics_csv=METRICS_CSV):
    """
    Select: typical / small tumor / large tumor / difficult case
    from the LDM-final results. Duplicates across categories are avoided.
    """
    print("\n" + "-" * 80)
    print("SELECTING FOUR REPRESENTATIVE CASES")
    print("-" * 80)

    df = pd.read_csv(metrics_csv)
    ldm = df[df["method"] == "LDM final (scaled_linear)"].copy()

    ldm["tumor_size"] = [
        int(np.sum(_load_generated(f"{LDM_FINAL_DIR}/{key}.pt")["mask"] > 0))
        for key in tqdm(ldm["key"],
                        desc="Loading tumor masks for case selection", unit="case")
    ]
    ldm = ldm[ldm["tumor_size"] > 0]
    if len(ldm) == 0:
        raise RuntimeError("No tumor-containing cases found.")

    median_psnr = ldm["psnr"].median()
    ranked = {
        "Typical": ldm.assign(_d=(ldm["psnr"] - median_psnr).abs())
                      .sort_values("_d")["key"],
        "Small tumor": ldm.sort_values("tumor_size")["key"],
        "Large/complex tumor": ldm.sort_values("tumor_size", ascending=False)["key"],
        "Difficult case": ldm.sort_values("psnr")["key"],
    }

    selected, used = {}, set()
    for label, keys in ranked.items():
        key = next((k for k in keys if k not in used), keys.iloc[0])
        selected[label] = key
        used.add(key)

    print("\nSelected cases:")
    for label, key in selected.items():
        print(f"  {label:25s}: {key}")
    return selected


# ==================================================================
# FIGURE 2: MAIN METHOD-COMPARISON GALLERY
# ==================================================================

GALLERY_COLUMNS = [
    "T1ce", "Real FLAIR", "GAN", "TransUNet",
    "LDM Base (linear)", "LDM Final (scaled)", "LDM Final Error",
]


def plot_main_gallery(selected_keys, save_path=GALLERY_PATH):
    """
    n_cases x 7 panels. Tumor boxes are drawn on EVERY row and column
    (boxes of tiny tumors are padded up to BOX_MIN_SPAN so the
    "Small tumor" row always shows its box). Error maps use the shared
    color domain ERROR_VMIN..ERROR_VMAX.
    """
    print("\n" + "-" * 80)
    print("GENERATING MAIN GALLERY")
    print("-" * 80)

    n_rows = len(selected_keys)
    fig, axes = plt.subplots(n_rows, 7, figsize=(24, 4 * n_rows), squeeze=False)

    for row, (label, key) in enumerate(
        tqdm(selected_keys.items(), desc="Generating main gallery", unit="case")
    ):
        ldm_final, ldm_base, gan, trans = load_all_methods(key)
        mask = ldm_final["mask"]

        panels = [
            ldm_final["real_t1ce"], ldm_final["real_flair"],
            gan["synthetic_flair"], trans["synthetic_flair"],
            ldm_base["synthetic_flair"], ldm_final["synthetic_flair"],
        ]

        # --- image panels: tumor box on every row (incl. row 2) ---
        for col, img in enumerate(panels):
            ax = axes[row, col]
            ax.imshow(img, cmap="gray")
            draw_tumor_box(ax, mask)
            ax.axis("off")
            if row == 0:
                ax.set_title(GALLERY_COLUMNS[col], fontsize=11)

        # --- error map: same color domain as Figure 3 difference maps ---
        err = np.abs(ldm_final["real_flair"] - ldm_final["synthetic_flair"])
        ax = axes[row, 6]
        im = show_error_map(
            ax, err, mask=mask,
            title=GALLERY_COLUMNS[6] if row == 0 else None,
        )
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        # --- row label ---
        axes[row, 0].text(
            -0.15, 0.5, label, transform=axes[row, 0].transAxes,
            rotation=90, va="center", ha="center",
            fontsize=12, fontweight="bold",
        )

    plt.tight_layout()
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


# ==================================================================
# FIGURE 3: CONDITIONING COMPARISON
# ==================================================================

COND_COLUMNS = [
    "Correct T1ce", "Wrong-patient T1ce\n(different tumor)", "Real FLAIR",
    "Output (correct cond)", "Output (wrong cond)", "Difference",
]


def plot_conditioning_comparison(pair_indices, val_dataset, seg_lookup,
                                 unet, scheduler, LATENT_MEAN, LATENT_STD,
                                 vae, device, guidance=GUIDANCE, eta=ETA,
                                 steps=DDIM_STEPS, save_path=COND_FIG_PATH):
    """
    For each target sample: condition on its own T1ce and on the most
    different other patient's T1ce, from identical starting noise.

    Boxes:
        solid red    = target patient's tumor
        dashed orange = wrong-condition patient's tumor
    Difference maps use the same color domain as Figure 2 error maps.
    """
    print("\n" + "-" * 80)
    print("GENERATING CONDITIONING COMPARISON FIGURE")
    print("-" * 80)

    tumor_stats = precompute_tumor_stats(seg_lookup)

    # Resolve pairs up front so no figure row ends up empty.
    pairs = []
    for i in pair_indices:
        j = find_most_different_tumor(i, tumor_stats)
        if j is None:
            print(f"  WARNING: no mismatched tumor found for idx {i}, skipping.")
        else:
            pairs.append((i, j))

    if not pairs:
        print("  No valid pairs - figure skipped.")
        return

    n_pairs = len(pairs)
    fig, axes = plt.subplots(n_pairs, 6, figsize=(20, 4 * n_pairs), squeeze=False)

    for row, (i, j) in enumerate(
        tqdm(pairs, desc="Generating conditioning figure", unit="pair")
    ):
        print(f"\n  Pair {row + 1}/{n_pairs}: "
              f"correct condition idx={i}, wrong condition idx={j}")

        imgs = sample_correct_wrong_pair(
            i, j, val_dataset, unet, scheduler, vae,
            LATENT_MEAN, LATENT_STD, device,
            guidance=guidance, eta=eta, steps=steps,
        )

        mask_target = seg_lookup[i]
        mask_wrong = seg_lookup[j]

        # (image, boxes to draw on it)
        panels = [
            (imgs["t1ce_correct"], [(mask_target, BOX_TARGET, "-")]),
            (imgs["t1ce_wrong"],   [(mask_wrong, BOX_WRONG, "--")]),
            (imgs["real_flair"],   [(mask_target, BOX_TARGET, "-")]),
            (imgs["out_correct"],  [(mask_target, BOX_TARGET, "-")]),
            (imgs["out_wrong"],    [(mask_target, BOX_TARGET, "-"),
                                    (mask_wrong, BOX_WRONG, "--")]),
        ]

        for col, (img, boxes) in enumerate(panels):
            ax = axes[row, col]
            ax.imshow(img, cmap="gray")
            for mask, color, ls in boxes:
                draw_tumor_box(ax, mask, color=color, linestyle=ls)
            ax.axis("off")
            if row == 0:
                ax.set_title(COND_COLUMNS[col], fontsize=10)

        # --- difference map: same color domain as Figure 2 error maps ---
        diff = np.abs(imgs["out_correct"] - imgs["out_wrong"])
        ax = axes[row, 5]
        im = show_error_map(
            ax, diff, mask=mask_target,
            title=COND_COLUMNS[5] if row == 0 else None,
        )
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        # --- row label ---
        axes[row, 0].text(
            -0.15, 0.5, f"Pair {row + 1}\n(idx {i} vs {j})",
            transform=axes[row, 0].transAxes, rotation=90,
            va="center", ha="center", fontsize=11, fontweight="bold",
        )

    # Legend explaining box styles
    handles = [
        Line2D([], [], color=BOX_TARGET, lw=2, label="Target patient tumor"),
        Line2D([], [], color=BOX_WRONG, lw=2, linestyle="--",
               label="Wrong-condition patient tumor"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


# ==================================================================
# PIPELINE
# ==================================================================

print("\n" + "=" * 80)
print("LOADING VALIDATION DATASET")
print("=" * 80)

val_dataset = PairedLatentDataset(VAL_PAIRS)
print(f"Validation samples: {len(val_dataset)}")

print("\nBuilding segmentation mask lookup...")
seg_lookup = build_seg_lookup(val_dataset, DATA_ROOT)
print(f"Done. {len(seg_lookup)} masks indexed.")

print("\n" + "=" * 80)
print("STARTING EVALUATION PIPELINE")
print(f"Device: {device} | dataset size: {len(val_dataset)} "
      f"| masks: {len(seg_lookup)}")
print("=" * 80)


# ---- Step 1/3: conditioning sensitivity test ----------------------
print("\n" + "=" * 80)
print("STEP 1/3: IMPROVED CONDITIONING TEST")
print("=" * 80)

df_cond = run_improved_conditioning_test(
    unet, ddim_scheduler, val_dataset, seg_lookup,
    LATENT_MEAN, LATENT_STD, vae, device,
)


# ---- Step 2/3: main gallery ----------------------------------------
print("\n" + "=" * 80)
print("STEP 2/3: MAIN GALLERY")
print("=" * 80)

selected = select_four_cases()
plot_main_gallery(selected)


# ---- Step 3/3: conditioning comparison ------------------------------
print("\n" + "=" * 80)
print("STEP 3/3: CONDITIONING COMPARISON")
print("=" * 80)

valid = [i for i, m in seg_lookup.items() if np.sum(m > 0) > 100]
pair_indices = valid[:2]
print(f"Available samples with tumor > 100 px: {len(valid)}")
print(f"Using pair indices: {pair_indices}")

plot_conditioning_comparison(
    pair_indices, val_dataset, seg_lookup,
    unet, ddim_scheduler, LATENT_MEAN, LATENT_STD, vae, device,
)


print("\n" + "=" * 80)
print("ALL EVALUATIONS COMPLETED")
print("=" * 80)
print("Generated files:")
print(f"  - {COND_CSV_PATH}")
print(f"  - {GALLERY_PATH}")
print(f"  - {COND_FIG_PATH}")
print("=" * 80)

In [ ]:


import numpy as np
import pandas as pd
import torch
import nibabel as nib
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from pathlib import Path
from scipy.stats import ttest_rel
from tqdm.auto import tqdm

from src.latent_dataset import PairedLatentDataset


# ==================================================================
# CONFIG
# ==================================================================

DATA_ROOT = Path(
    "BraTS2020_TrainingData/"
    "BraTS2020_TrainingData/"
    "MICCAI_BraTS2020_TrainingData"
)

VAL_PAIRS = "latent_pairs/val_pairs_scaled.pt"

# Generated-sample locations (per case/slice key)
LDM_FINAL_DIR = "evaluation_results_scaled_tuned/generated_samples"
LDM_BASE_DIR = "evaluation_results_Linear/generated_samples"
GAN_DIR = "pix2pix_baseline"
TRANS_DIR = "transformer_full_val_eval"

# Outputs
METRICS_CSV = "fair_comparison_raw_values.csv"
COND_CSV_PATH = "conditioning_test_improved.csv"
GALLERY_PATH = "figure2_main_gallery.png"
COND_FIG_PATH = "figure3_conditioning_comparison.png"

# Sampling settings
GUIDANCE = 2.6
ETA = 0.0                  # deterministic DDIM
DDIM_STEPS = 200
NUM_COND_PAIRS = 25
MIN_TUMOR_SIZE = 50        # px required for conditioning-test eligibility
NOISE_SEED_BASE = 1234
RNG_SEED = 42

# --- Matched wrong-condition selection -----------------------------
SLICE_TOL = 3              # anatomical-level tolerance (slices)
MATCH_QUANTILE = 0.25      # keep closest 25% by NON-TUMOR latent distance
MIN_TUMOR_DIST_PX = 15     # required minimum tumor centroid displacement

# --- Visualization settings -----------------------------------------
IMG_VMIN = 0.0             # IDENTICAL display window for every gray panel
IMG_VMAX = 1.0             # (fixes per-panel auto-normalization!)
ERROR_VMIN = 0.0           # shared color domain for ALL error/difference maps
ERROR_VMAX = 0.3
BOX_TARGET = "red"         # solid: target patient's tumor
BOX_WRONG = "orange"       # dashed: wrong-condition patient's tumor
BOX_ON_ERROR = "cyan"
BOX_MIN_SPAN = 10          # px, keeps small-tumor boxes visible
FIG_DPI = 200


# ==================================================================
# SEGMENTATION MASKS + METADATA + TUMOR STATISTICS
# ==================================================================

def build_seg_lookup(val_dataset, data_root):
    """
    Returns:
        lookup : index -> 2D tumor mask (160x160 crop used by the dataset)
        meta   : index -> {"case": str, "slice": int}
    """
    lookup, seg_cache, meta = {}, {}, {}

    for i in tqdm(range(len(val_dataset)),
                  desc="Building segmentation lookup", unit="slice"):
        sample = val_dataset[i]
        case, slice_idx = sample["case"], sample["slice_idx"]
        meta[i] = {"case": case, "slice": int(slice_idx)}

        if case not in seg_cache:
            seg_file = data_root / case / f"{case}_seg.nii"
            seg_cache[case] = (
                nib.load(seg_file).get_fdata() if seg_file.exists() else None
            )

        seg_vol = seg_cache[case]
        if seg_vol is not None and slice_idx < seg_vol.shape[2]:
            lookup[i] = seg_vol[48:208, 48:208, slice_idx]
        else:
            lookup[i] = np.zeros((160, 160), dtype=np.float32)

    return lookup, meta


def tumor_centroid_and_size(mask):
    """-> (centroid [y, x], tumor pixel count). Falls back to image center."""
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return np.array([80.0, 80.0], dtype=np.float32), 0
    return np.array([ys.mean(), xs.mean()], dtype=np.float32), len(ys)


def precompute_tumor_stats(seg_lookup):
    """index -> (centroid, size), computed once and reused everywhere."""
    return {i: tumor_centroid_and_size(m) for i, m in seg_lookup.items()}


# ==================================================================
# MATCHED WRONG-CONDITION SELECTION
# ==================================================================

def to_latent_mask(mask, latent_hw):
    """Downsample a 160x160 tumor mask into latent spatial resolution."""
    h_l, w_l = latent_hw
    fh = mask.shape[0] // h_l
    fw = mask.shape[1] // w_l
    crop = mask[:h_l * fh, :w_l * fw]
    m = crop.reshape(h_l, fh, w_l, fw).mean(axis=(1, 3))
    return torch.from_numpy(m > 0.25)


def precompute_condition_cache(val_dataset, seg_lookup):
    """
    Cache, once, for every validation sample:
        cond_latents : index -> normalized T1ce condition latent (CPU)
        latent_masks : index -> tumor mask in latent resolution
    Used to measure GLOBAL (non-tumor) appearance similarity between
    candidate conditions.
    """
    cond_latents, latent_masks = {}, {}
    latent_hw = None

    for i in tqdm(range(len(val_dataset)),
                  desc="Caching condition latents", unit="sample"):
        c = val_dataset[i]["condition"].detach().cpu()
        cond_latents[i] = c
        if latent_hw is None:
            latent_hw = tuple(c.shape[-2:])
        latent_masks[i] = to_latent_mask(seg_lookup[i], latent_hw)

    return cond_latents, latent_masks


def nontumor_latent_dist(c_i, c_j, m_i, m_j):
    """
    MSE between two condition latents measured ONLY outside both tumor
    regions -> pure global appearance/intensity similarity, independent
    of the tumor itself.
    """
    outside = ~(m_i | m_j)
    if outside.sum() < 8:
        outside = ~m_i
    keep = outside.unsqueeze(0).expand_as(c_i)
    return float(((c_i - c_j)[keep] ** 2).mean())


def find_matched_wrong_condition(target_idx, meta, tumor_stats,
                                 cond_latents, latent_masks,
                                 min_size=MIN_TUMOR_SIZE,
                                 slice_tol=SLICE_TOL,
                                 match_quantile=MATCH_QUANTILE,
                                 min_tumor_dist=MIN_TUMOR_DIST_PX):
    """
    Pick a WRONG condition that is globally similar to the target but
    differs in tumor location/size:

      1. different patient,
      2. similar anatomical level (|slice_i - slice_j| <= slice_tol,
         progressively relaxed if no candidate exists),
      3. tumor size >= min_size,
      4. global appearance similar: non-tumor latent distance within the
         lowest `match_quantile` of all candidates,
      5. among those, the one with the largest tumor displacement
         (must be >= min_tumor_dist px when possible).

    -> (best_j, info_dict) or (None, {})
    """
    t_centroid, t_size = tumor_stats[target_idx]
    if t_size == 0:
        return None, {}

    t_meta = meta[target_idx]
    t_lat, t_lmask = cond_latents[target_idx], latent_masks[target_idx]

    # 1-3) candidates, with progressive relaxation of slice tolerance
    candidates = []
    for tol in (slice_tol, 2 * slice_tol, 4 * slice_tol, 10_000):
        candidates = [
            j for j, (cent, size) in tumor_stats.items()
            if j != target_idx
            and meta[j]["case"] != t_meta["case"]
            and size >= min_size
            and abs(meta[j]["slice"] - t_meta["slice"]) <= tol
        ]
        if candidates:
            break
    if not candidates:
        return None, {}

    # 4) global (non-tumor) appearance distances
    dists = {
        j: nontumor_latent_dist(t_lat, cond_latents[j], t_lmask, latent_masks[j])
        for j in candidates
    }
    thresh = np.quantile(list(dists.values()), match_quantile)
    matched = [j for j, d in dists.items() if d <= max(thresh, 1e-8)]

    # 5) among matched candidates, maximize tumor displacement
    def tumor_diff(j):
        cent, size = tumor_stats[j]
        return float(np.linalg.norm(t_centroid - cent) + abs(t_size - size) / 100.0)

    pool = [
        j for j in matched
        if np.linalg.norm(t_centroid - tumor_stats[j][0]) >= min_tumor_dist
    ]
    if not pool:
        pool = matched          # fallback: best global match, smaller displacement
    if not pool:
        pool = candidates       # last resort

    best = max(pool, key=tumor_diff)

    info = {
        "nontumor_latent_dist": dists[best],
        "slice_diff": abs(meta[best]["slice"] - t_meta["slice"]),
        "n_candidates": len(candidates),
        "n_matched": len(matched),
    }
    return best, info


# ==================================================================
# DRAWING HELPERS
# ==================================================================

def draw_tumor_box(ax, mask, color=BOX_TARGET, linewidth=2.0,
                   linestyle="-", min_span=BOX_MIN_SPAN):
    """Bounding box around the tumor, padded up to `min_span` px so that
    tiny tumors stay visible."""
    ys, xs = np.where(mask > 0)
    if len(ys) == 0:
        return

    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    if (y1 - y0) < min_span:
        pad = (min_span - (y1 - y0)) / 2
        y0, y1 = y0 - pad, y1 + pad
    if (x1 - x0) < min_span:
        pad = (min_span - (x1 - x0)) / 2
        x0, x1 = x0 - pad, x1 + pad

    ax.add_patch(
        Rectangle((x0, y0), x1 - x0, y1 - y0,
                  linewidth=linewidth, linestyle=linestyle,
                  edgecolor=color, facecolor="none")
    )


def show_gray(ax, img, title=None):
    """
    Display a gray-scale image with the FIXED window IMG_VMIN..IMG_VMAX.

    NOTE: without explicit vmin/vmax, matplotlib auto-normalizes EACH
    panel to its own data range, which would make two images with
    identical preprocessing look different in brightness/contrast.
    """
    im = ax.imshow(img, cmap="gray", vmin=IMG_VMIN, vmax=IMG_VMAX)
    if title:
        ax.set_title(title, fontsize=10)
    ax.axis("off")
    return im


def show_error_map(ax, err, mask=None, title=None):
    """Error/difference map on the SHARED color domain (ERROR_VMIN..ERROR_VMAX)."""
    im = ax.imshow(err, cmap="hot", vmin=ERROR_VMIN, vmax=ERROR_VMAX)
    if mask is not None:
        draw_tumor_box(ax, mask, color=BOX_ON_ERROR)
    if title:
        ax.set_title(title, fontsize=10)
    ax.axis("off")
    return im


# ==================================================================
# DIFFUSION SAMPLING
# ==================================================================

@torch.no_grad()
def sample_with_condition(unet, scheduler, condition, start_latent,
                          guidance=GUIDANCE, eta=ETA, steps=DDIM_STEPS,
                          progress_desc="DDIM sampling"):
    """Classifier-free-guided DDIM sampling (deterministic when eta=0)."""
    scheduler.set_timesteps(steps)
    latent = start_latent.clone()

    for t in tqdm(scheduler.timesteps, desc=progress_desc, unit="step", leave=False):
        timestep = torch.full((latent.shape[0],), int(t),
                              device=latent.device, dtype=torch.long)

        noise_cond = unet(torch.cat([latent, condition], dim=1), timestep)
        noise_uncond = unet(
            torch.cat([latent, torch.zeros_like(condition)], dim=1), timestep
        )
        noise_pred = noise_uncond + guidance * (noise_cond - noise_uncond)

        latent, _ = scheduler.step(noise_pred, int(t), latent, eta=eta)

    return latent


def to_img(z, LATENT_MEAN, LATENT_STD, vae):
    """
    Normalized latent -> [0, 1] numpy image.

    EVERY displayed image in this script goes through this exact same
    function (same VAE decode, same clamp, same rescale), so pixel
    intensity spaces are identical across correct/wrong conditions.
    """
    img = vae.decode(z * LATENT_STD + LATENT_MEAN)
    img = (torch.clamp(img, -1, 1) + 1) / 2
    return img.squeeze().detach().cpu().numpy()


def check_display_consistency(images):
    """
    Sanity check: all images must lie in [0, 1]. Combined with the fixed
    imshow window (IMG_VMIN/IMG_VMAX) this guarantees IDENTICAL display
    windowing for correct and wrong conditions.
    """
    for name, im in images.items():
        if im.min() < -1e-6 or im.max() > 1 + 1e-6:
            raise ValueError(
                f"Display consistency violated for '{name}': "
                f"range [{im.min():.4f}, {im.max():.4f}] is outside [0, 1]."
            )


def sample_correct_wrong_pair(target_idx, wrong_idx, val_dataset,
                              unet, scheduler, vae,
                              LATENT_MEAN, LATENT_STD, device,
                              guidance=GUIDANCE, eta=ETA, steps=DDIM_STEPS):
    """
    Sample twice from the SAME starting noise (correct vs. wrong condition)
    and decode everything needed for metrics and figures.
    """
    sample = val_dataset[target_idx]
    cond_correct = sample["condition"].unsqueeze(0).to(device)
    target = sample["target"].unsqueeze(0).to(device)
    cond_wrong = val_dataset[wrong_idx]["condition"].unsqueeze(0).to(device)

    torch.manual_seed(NOISE_SEED_BASE + int(target_idx))
    start_noise = torch.randn_like(target)

    z_correct = sample_with_condition(unet, scheduler, cond_correct, start_noise,
                                      guidance, eta, steps, "  DDIM correct")
    z_wrong = sample_with_condition(unet, scheduler, cond_wrong, start_noise,
                                    guidance, eta, steps, "  DDIM wrong")

    imgs = {
        "t1ce_correct": to_img(cond_correct, LATENT_MEAN, LATENT_STD, vae),
        "t1ce_wrong": to_img(cond_wrong, LATENT_MEAN, LATENT_STD, vae),
        "real_flair": to_img(target, LATENT_MEAN, LATENT_STD, vae),
        "out_correct": to_img(z_correct, LATENT_MEAN, LATENT_STD, vae),
        "out_wrong": to_img(z_wrong, LATENT_MEAN, LATENT_STD, vae),
    }
    check_display_consistency(imgs)
    return imgs


def region_split_error(real_img, synth_img, mask):
    """-> (whole-image MAE, tumor-region MAE). Tumor MAE is NaN if no tumor."""
    whole_mae = np.mean(np.abs(real_img - synth_img))
    if np.sum(mask > 0) > 0:
        tumor_mae = np.mean(np.abs(real_img[mask > 0] - synth_img[mask > 0]))
    else:
        tumor_mae = np.nan
    return whole_mae, tumor_mae


# ==================================================================
# PART A: CONDITIONING SENSITIVITY TEST (MATCHED WRONG CONDITION)
# ==================================================================

def run_improved_conditioning_test(unet, scheduler, val_dataset, seg_lookup,
                                   meta, tumor_stats,
                                   cond_latents, latent_masks,
                                   LATENT_MEAN, LATENT_STD, vae, device,
                                   num_pairs=NUM_COND_PAIRS, eta=ETA,
                                   guidance=GUIDANCE, steps=DDIM_STEPS,
                                   min_tumor_size=MIN_TUMOR_SIZE):
    """
    For each selected sample: generate with (a) the correct condition and
    (b) a MATCHED wrong condition (same anatomical level + similar global
    appearance, different tumor), from identical starting noise.
    """
    print("\n" + "-" * 80)
    print("PREPARING CONDITIONING SENSITIVITY TEST (matched wrong condition)")
    print("-" * 80)

    valid_indices = [
        i for i, (_, size) in tumor_stats.items() if size >= min_tumor_size
    ]
    print(f"Valid samples with tumor >= {min_tumor_size} px: {len(valid_indices)}")
    if not valid_indices:
        raise RuntimeError("No valid samples found for conditioning test.")

    rng = np.random.RandomState(RNG_SEED)
    test_indices = rng.choice(
        valid_indices, size=min(num_pairs, len(valid_indices)), replace=False
    )
    print(f"Testing {len(test_indices)} pairs...")

    rows = []
    for pair_no, i in enumerate(
        tqdm(test_indices, desc="Conditioning sensitivity test", unit="pair"), 1
    ):
        j, info = find_matched_wrong_condition(
            i, meta, tumor_stats, cond_latents, latent_masks,
            min_size=min_tumor_size,
        )
        if j is None:
            print(f"  Pair {pair_no}: no matched wrong condition for idx {i}, skipping.")
            continue

        imgs = sample_correct_wrong_pair(
            i, j, val_dataset, unet, scheduler, vae,
            LATENT_MEAN, LATENT_STD, device,
            guidance=guidance, eta=eta, steps=steps,
        )

        mask = seg_lookup[i]
        real = imgs["real_flair"]
        whole_c, tumor_c = region_split_error(real, imgs["out_correct"], mask)
        whole_w, tumor_w = region_split_error(real, imgs["out_wrong"], mask)

        t_centroid, t_size = tumor_stats[i]
        w_centroid, w_size = tumor_stats[j]

        rows.append({
            "idx": int(i),
            "wrong_idx": int(j),
            "mse_correct": float(np.mean((imgs["out_correct"] - real) ** 2)),
            "mse_wrong": float(np.mean((imgs["out_wrong"] - real) ** 2)),
            "whole_mae_correct": whole_c,
            "whole_mae_wrong": whole_w,
            "tumor_mae_correct": tumor_c,
            "tumor_mae_wrong": tumor_w,
            "tumor_centroid_distance_px": float(np.linalg.norm(t_centroid - w_centroid)),
            "tumor_size_diff_px": float(abs(t_size - w_size)),
            # matching-quality diagnostics (for the thesis)
            "nontumor_latent_dist": info["nontumor_latent_dist"],
            "slice_diff": info["slice_diff"],
        })

        print(
            f"  Pair {pair_no}/{len(test_indices)}: idx {i} vs {j} "
            f"(d_global={info['nontumor_latent_dist']:.4f}, "
            f"d_slice={info['slice_diff']}) | "
            f"whole MAE {whole_c:.4f} vs {whole_w:.4f} | "
            f"tumor MAE {tumor_c:.4f} vs {tumor_w:.4f}"
        )

    df = pd.DataFrame(rows)
    df.to_csv(COND_CSV_PATH, index=False)
    print(f"\nResults saved to: {COND_CSV_PATH}")

    _print_conditioning_summary(df, eta)
    return df


def _print_conditioning_summary(df, eta):
    print("\n" + "=" * 80)
    print(f"IMPROVED CONDITIONING TEST (n={len(df)}, eta={eta}, MATCHED wrong condition)")
    print("=" * 80)

    if len(df) == 0:
        print("No valid test pairs were produced.")
        return

    metric_cols = [
        "mse_correct", "mse_wrong",
        "whole_mae_correct", "whole_mae_wrong",
        "tumor_mae_correct", "tumor_mae_wrong",
        "tumor_centroid_distance_px", "tumor_size_diff_px",
        "nontumor_latent_dist", "slice_diff",
    ]
    print(df[metric_cols].describe())

    tumor = df.dropna(subset=["tumor_mae_correct", "tumor_mae_wrong"])
    if len(tumor) >= 2:
        t, p = ttest_rel(tumor["tumor_mae_correct"], tumor["tumor_mae_wrong"])
        print(f"\nPaired t-test, TUMOR REGION MAE (correct vs wrong): "
              f"t={t:.3f}, p={p:.4f}")
    else:
        print("\nNot enough valid tumor-region pairs for tumor MAE t-test.")

    t, p = ttest_rel(df["whole_mae_correct"], df["whole_mae_wrong"])
    print(f"\nPaired t-test, WHOLE IMAGE MAE (correct vs wrong): "
          f"t={t:.3f}, p={p:.4f}")

    print("\n--- Matching quality of wrong conditions ---")
    print(f"Mean tumor centroid distance (large, as intended): "
          f"{df['tumor_centroid_distance_px'].mean():.1f} px")
    print(f"Mean tumor size difference: "
          f"{df['tumor_size_diff_px'].mean():.1f} px")
    print(f"Mean NON-TUMOR latent distance (should be small): "
          f"{df['nontumor_latent_dist'].mean():.4f}")
    print(f"Mean slice difference (should be small): "
          f"{df['slice_diff'].mean():.2f}")


# ==================================================================
# PART B: LOAD ALL METHODS FOR ONE CASE
# ==================================================================

def _load_generated(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing generated sample: {path}")
    return torch.load(path, weights_only=False)


def load_all_methods(key):
    """-> (ldm_final, ldm_base, gan, trans) dicts for one case/slice key."""
    return (
        _load_generated(f"{LDM_FINAL_DIR}/{key}.pt"),
        _load_generated(f"{LDM_BASE_DIR}/{key}.pt"),
        _load_generated(f"{GAN_DIR}/{key}.pt"),
        _load_generated(f"{TRANS_DIR}/{key}.pt"),
    )


def select_four_cases(metrics_csv=METRICS_CSV):
    """Typical / small tumor / large tumor / difficult case (no duplicates)."""
    print("\n" + "-" * 80)
    print("SELECTING FOUR REPRESENTATIVE CASES")
    print("-" * 80)

    df = pd.read_csv(metrics_csv)
    ldm = df[df["method"] == "LDM final (scaled_linear)"].copy()

    ldm["tumor_size"] = [
        int(np.sum(_load_generated(f"{LDM_FINAL_DIR}/{key}.pt")["mask"] > 0))
        for key in tqdm(ldm["key"],
                        desc="Loading tumor masks for case selection", unit="case")
    ]
    ldm = ldm[ldm["tumor_size"] > 0]
    if len(ldm) == 0:
        raise RuntimeError("No tumor-containing cases found.")

    median_psnr = ldm["psnr"].median()
    ranked = {
        "Typical": ldm.assign(_d=(ldm["psnr"] - median_psnr).abs())
                      .sort_values("_d")["key"],
        "Small tumor": ldm.sort_values("tumor_size")["key"],
        "Large/complex tumor": ldm.sort_values("tumor_size", ascending=False)["key"],
        "Difficult case": ldm.sort_values("psnr")["key"],
    }

    selected, used = {}, set()
    for label, keys in ranked.items():
        key = next((k for k in keys if k not in used), keys.iloc[0])
        selected[label] = key
        used.add(key)

    print("\nSelected cases:")
    for label, key in selected.items():
        print(f"  {label:25s}: {key}")
    return selected


# ==================================================================
# FIGURE 2: MAIN METHOD-COMPARISON GALLERY
# ==================================================================

GALLERY_COLUMNS = [
    "T1ce", "Real FLAIR", "GAN", "TransUNet",
    "LDM Base (linear)", "LDM Final (scaled)", "LDM Final Error",
]


def plot_main_gallery(selected_keys, save_path=GALLERY_PATH):
    """n_cases x 7 panels, tumor box on every row, fixed display window."""
    print("\n" + "-" * 80)
    print("GENERATING MAIN GALLERY")
    print("-" * 80)

    n_rows = len(selected_keys)
    fig, axes = plt.subplots(n_rows, 7, figsize=(24, 4 * n_rows), squeeze=False)

    for row, (label, key) in enumerate(
        tqdm(selected_keys.items(), desc="Generating main gallery", unit="case")
    ):
        ldm_final, ldm_base, gan, trans = load_all_methods(key)
        mask = ldm_final["mask"]

        panels = [
            ldm_final["real_t1ce"], ldm_final["real_flair"],
            gan["synthetic_flair"], trans["synthetic_flair"],
            ldm_base["synthetic_flair"], ldm_final["synthetic_flair"],
        ]

        for col, img in enumerate(panels):
            ax = axes[row, col]
            show_gray(ax, img,
                      title=GALLERY_COLUMNS[col] if row == 0 else None)
            draw_tumor_box(ax, mask)

        err = np.abs(ldm_final["real_flair"] - ldm_final["synthetic_flair"])
        ax = axes[row, 6]
        im = show_error_map(ax, err, mask=mask,
                            title=GALLERY_COLUMNS[6] if row == 0 else None)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        axes[row, 0].text(
            -0.15, 0.5, label, transform=axes[row, 0].transAxes,
            rotation=90, va="center", ha="center",
            fontsize=12, fontweight="bold",
        )

    plt.tight_layout()
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


# ==================================================================
# FIGURE 3: CONDITIONING COMPARISON (MATCHED WRONG CONDITION)
# ==================================================================

COND_COLUMNS = [
    "Correct T1ce", "Wrong-patient T1ce\n(matched appearance)",
    "T1ce difference", "Real FLAIR",
    "Output (correct cond)", "Output (wrong cond)", "Difference",
]


def plot_conditioning_comparison(pair_indices, val_dataset, seg_lookup,
                                 meta, tumor_stats,
                                 cond_latents, latent_masks,
                                 unet, scheduler, LATENT_MEAN, LATENT_STD,
                                 vae, device, guidance=GUIDANCE, eta=ETA,
                                 steps=DDIM_STEPS, save_path=COND_FIG_PATH):
    """
    Matched-conditioning visualization, 7 columns:

        Correct T1ce | Wrong (matched) T1ce | T1ce difference |
        Real FLAIR | Output (correct) | Output (wrong) | Output difference

    The "T1ce difference" panel (same shared color scale as the output
    difference) visually demonstrates that the two conditions differ
    mainly at the tumor, not in global intensity/anatomy.
    """
    print("\n" + "-" * 80)
    print("GENERATING CONDITIONING COMPARISON FIGURE")
    print("-" * 80)

    # Resolve MATCHED pairs up front so no figure row ends up empty.
    pairs = []
    for i in pair_indices:
        j, info = find_matched_wrong_condition(i, meta, tumor_stats,
                                               cond_latents, latent_masks)
        if j is None:
            print(f"  WARNING: no matched wrong condition for idx {i}, skipping.")
        else:
            pairs.append((i, j, info))
            print(f"  Pair: target idx={i} <- wrong idx={j} "
                  f"(d_global={info['nontumor_latent_dist']:.4f}, "
                  f"d_slice={info['slice_diff']}, "
                  f"candidates={info['n_candidates']}, "
                  f"matched={info['n_matched']})")

    if not pairs:
        print("  No valid pairs - figure skipped.")
        return

    n_pairs = len(pairs)
    fig, axes = plt.subplots(n_pairs, 7, figsize=(24, 4 * n_pairs), squeeze=False)

    for row, (i, j, info) in enumerate(
        tqdm(pairs, desc="Generating conditioning figure", unit="pair")
    ):
        print(f"\n  Pair {row + 1}/{n_pairs}: correct idx={i}, wrong idx={j}")

        imgs = sample_correct_wrong_pair(
            i, j, val_dataset, unet, scheduler, vae,
            LATENT_MEAN, LATENT_STD, device,
            guidance=guidance, eta=eta, steps=steps,
        )

        mask_target = seg_lookup[i]
        mask_wrong = seg_lookup[j]

        gray_panels = {
            0: (imgs["t1ce_correct"], [(mask_target, BOX_TARGET, "-")]),
            1: (imgs["t1ce_wrong"],   [(mask_wrong, BOX_WRONG, "--")]),
            3: (imgs["real_flair"],   [(mask_target, BOX_TARGET, "-")]),
            4: (imgs["out_correct"],  [(mask_target, BOX_TARGET, "-")]),
            5: (imgs["out_wrong"],    [(mask_target, BOX_TARGET, "-"),
                                       (mask_wrong, BOX_WRONG, "--")]),
        }

        for col, (img, boxes) in gray_panels.items():
            ax = axes[row, col]
            show_gray(ax, img, title=COND_COLUMNS[col] if row == 0 else None)
            for mask, color, ls in boxes:
                draw_tumor_box(ax, mask, color=color, linestyle=ls)

        # --- condition difference: proves the mismatch is local (tumor) ---
        cond_diff = np.abs(imgs["t1ce_correct"] - imgs["t1ce_wrong"])
        im_cond = show_error_map(
            axes[row, 2], cond_diff, mask=mask_target,
            title=COND_COLUMNS[2] if row == 0 else None,
        )
        fig.colorbar(im_cond, ax=axes[row, 2], fraction=0.046, pad=0.04)

        # --- output difference: same shared color scale -------------------
        out_diff = np.abs(imgs["out_correct"] - imgs["out_wrong"])
        im_out = show_error_map(
            axes[row, 6], out_diff, mask=mask_target,
            title=COND_COLUMNS[6] if row == 0 else None,
        )
        fig.colorbar(im_out, ax=axes[row, 6], fraction=0.046, pad=0.04)

        # --- row label with matching diagnostics ---------------------------
        axes[row, 0].text(
            -0.18, 0.5,
            f"Pair {row + 1} (idx {i} vs {j})\n"
            f"d_slice={info['slice_diff']}, "
            f"d_global={info['nontumor_latent_dist']:.3f}",
            transform=axes[row, 0].transAxes, rotation=90,
            va="center", ha="center", fontsize=10, fontweight="bold",
        )

    handles = [
        Line2D([], [], color=BOX_TARGET, lw=2, label="Target patient tumor"),
        Line2D([], [], color=BOX_WRONG, lw=2, linestyle="--",
               label="Wrong-condition patient tumor"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


# ==================================================================
# PIPELINE
# ==================================================================

print("\n" + "=" * 80)
print("LOADING VALIDATION DATASET")
print("=" * 80)

val_dataset = PairedLatentDataset(VAL_PAIRS)
print(f"Validation samples: {len(val_dataset)}")

print("\nBuilding segmentation mask lookup...")
seg_lookup, meta = build_seg_lookup(val_dataset, DATA_ROOT)
print(f"Done. {len(seg_lookup)} masks indexed.")

print("\nPrecomputing tumor statistics + condition-latent cache...")
tumor_stats = precompute_tumor_stats(seg_lookup)
cond_latents, latent_masks = precompute_condition_cache(val_dataset, seg_lookup)

print("\n" + "=" * 80)
print("STARTING EVALUATION PIPELINE")
print(f"Device: {device} | dataset size: {len(val_dataset)} "
      f"| masks: {len(seg_lookup)}")
print("=" * 80)


# ---- Step 1/3: conditioning sensitivity test ----------------------
print("\n" + "=" * 80)
print("STEP 1/3: IMPROVED CONDITIONING TEST (matched wrong condition)")
print("=" * 80)

df_cond = run_improved_conditioning_test(
    unet, ddim_scheduler, val_dataset, seg_lookup,
    meta, tumor_stats, cond_latents, latent_masks,
    LATENT_MEAN, LATENT_STD, vae, device,
)


# ---- Step 2/3: main gallery ----------------------------------------
print("\n" + "=" * 80)
print("STEP 2/3: MAIN GALLERY")
print("=" * 80)

selected = select_four_cases()
plot_main_gallery(selected)


# ---- Step 3/3: conditioning comparison ------------------------------
print("\n" + "=" * 80)
print("STEP 3/3: CONDITIONING COMPARISON")
print("=" * 80)

valid = [i for i, m in seg_lookup.items() if np.sum(m > 0) > 100]
pair_indices = valid[:2]
print(f"Available samples with tumor > 100 px: {len(valid)}")
print(f"Using pair indices: {pair_indices}")

plot_conditioning_comparison(
    pair_indices, val_dataset, seg_lookup,
    meta, tumor_stats, cond_latents, latent_masks,
    unet, ddim_scheduler, LATENT_MEAN, LATENT_STD, vae, device,
)


print("\n" + "=" * 80)
print("ALL EVALUATIONS COMPLETED")
print("=" * 80)
print("Generated files:")
print(f"  - {COND_CSV_PATH}")
print(f"  - {GALLERY_PATH}")
print(f"  - {COND_FIG_PATH}")
print("=" * 80)

In [ ]:
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from scipy.stats import norm

try:
    import lpips
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    lpips_model = lpips.LPIPS(net='alex').to(device)
    lpips_model.eval()
    LPIPS_AVAILABLE = True
except ImportError:
    LPIPS_AVAILABLE = False
    print("lpips not installed -- LPIPS column will be skipped.")

# ==========================================
# CONFIG
# ==========================================
METHOD_FOLDERS = {
    "LDM baseline (linear)":        Path("evaluation_results_Linear/generated_samples"),
    "LDM baseline (scaled_linear)": Path("evaluation_results_scaled_basic/generated_samples"),
    "LDM final (scaled_linear)":    Path("evaluation_results_THESIS_FINAL_2/generated_samples"),
}

# ==========================================
# STEP 1: common keys across all methods, FILTERED to slice 80 only
# ==========================================
def get_available_keys(folder):
    return {f.stem for f in folder.glob("*.pt")}

key_sets = {name: get_available_keys(folder) for name, folder in METHOD_FOLDERS.items()}

# Get all common keys first
common_keys_all = set.intersection(*key_sets.values())

# Filter to only keys that end with "_slice_80" (or contain "slice80" depending on naming)
# Adjust the pattern based on your actual filename format
SLICE_NUMBER = 80
common_keys = {k for k in common_keys_all if f"slice_{SLICE_NUMBER}" in k or f"slice{SLICE_NUMBER}" in k or f"_s{SLICE_NUMBER}" in k}

# If no keys found with the pattern, try to extract from filenames more flexibly
if len(common_keys) == 0:
    # Try to find any keys that might contain slice 80 in various formats
    slice_patterns = [f"slice_{SLICE_NUMBER}", f"slice{SLICE_NUMBER}", f"_s{SLICE_NUMBER}", f"_{SLICE_NUMBER}_"]
    common_keys = set()
    for k in common_keys_all:
        for pattern in slice_patterns:
            if pattern in k:
                common_keys.add(k)
                break

print("Per-method file counts (total):")
for name, keys in key_sets.items():
    print(f"  {name}: {len(keys)} files")

# Count how many slice 80 files each method has
slice80_keys_all = set()
for name, folder in METHOD_FOLDERS.items():
    slice80_keys = {f.stem for f in folder.glob("*.pt") if f"slice_{SLICE_NUMBER}" in f.stem or f"slice{SLICE_NUMBER}" in f.stem or f"_s{SLICE_NUMBER}" in f.stem}
    slice80_keys_all.update(slice80_keys)
    print(f"  {name} (slice {SLICE_NUMBER} only): {len(slice80_keys)} files")

print(f"\nCommon (case, slice {SLICE_NUMBER}) keys across ALL methods: {len(common_keys)}")

if len(common_keys) == 0:
    # If still no keys, try to extract patient IDs and manually filter
    print(f"\nWARNING: No slice {SLICE_NUMBER} keys found. Checking available keys...")
    sample_keys = list(common_keys_all)[:10]
    print(f"Sample keys: {sample_keys}")
    print("Please adjust the slice filtering pattern based on your actual filename format.")
    raise RuntimeError(f"No overlapping samples with slice {SLICE_NUMBER} found across all methods.")

common_keys = sorted(common_keys)

# ==========================================
# STEP 2: shared metric functions (unchanged)
# ==========================================
def compute_all_metrics(real, synth):
    if real.shape != synth.shape:
        from skimage.transform import resize
        synth = resize(synth, real.shape)

    data_range = max(np.max(real) - np.min(real), 1e-6)

    try:
        p = psnr(real, synth, data_range=data_range)
    except Exception:
        p = np.nan
    try:
        win_size = min(7, min(real.shape) // 2)
        if win_size % 2 == 0:
            win_size -= 1
        s = ssim(real, synth, data_range=data_range, win_size=max(3, win_size))
    except Exception:
        s = np.nan

    mae = np.mean(np.abs(real - synth))

    l = np.nan
    if LPIPS_AVAILABLE:
        real_t = torch.from_numpy(real * 2 - 1).float().unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1).to(device)
        synth_t = torch.from_numpy(synth * 2 - 1).float().unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1).to(device)
        with torch.no_grad():
            l = lpips_model(real_t, synth_t).item()

    return p, s, l, mae

def calculate_clinical_metrics(real_img, gen_img, mask_img):
    """gCNR/d' computed on the GENERATED image, using the established (corrected) formula:
    gCNR uses z_score (un-pooled denominator); d'/gSNR uses the pooled-noise denominator.
    These are two DIFFERENT quantities -- do not substitute one for the other."""
    fg_mask = mask_img > 0
    bg_mask = mask_img == 0
    if np.sum(fg_mask) == 0 or np.sum(bg_mask) == 0:
        return np.nan, np.nan

    gen_fg = gen_img[fg_mask].flatten()
    gen_bg = gen_img[bg_mask].flatten()
    mu_fg, mu_bg = np.mean(gen_fg), np.mean(gen_bg)
    std_fg, std_bg = np.std(gen_fg), np.std(gen_bg)

    pooled_noise = np.sqrt(0.5 * (std_fg**2 + std_bg**2))
    if pooled_noise == 0:
        return np.nan, np.nan
    signal_diff = np.abs(mu_fg - mu_bg)

    d_prime = signal_diff / pooled_noise  # gSNR
    z_score = signal_diff / np.sqrt(std_fg**2 + std_bg**2)
    gcnr = 1 - 2 * norm.cdf(-z_score / 2)

    return gcnr, d_prime

# ==========================================
# STEP 3: run every method on the exact same common_keys (slice 80 only)
# ==========================================
results = {name: {'psnr': [], 'ssim': [], 'lpips': [], 'mae': [], 'gcnr': [], 'd_prime': []}
           for name in METHOD_FOLDERS}

missing_mask_warned = set()

for key in common_keys:
    for name, folder in METHOD_FOLDERS.items():
        d = torch.load(folder / f"{key}.pt", weights_only=False)
        real = np.asarray(d['real_flair'])
        synth = np.asarray(d['synthetic_flair'])

        p, s, l, mae = compute_all_metrics(real, synth)
        results[name]['psnr'].append(p)
        results[name]['ssim'].append(s)
        results[name]['lpips'].append(l)
        results[name]['mae'].append(mae)

        mask = d.get('mask', None)
        if mask is None:
            if name not in missing_mask_warned:
                print(f"WARNING: '{name}' has no 'mask' key -- gCNR/d' will be NaN for all its samples.")
                missing_mask_warned.add(name)
            gcnr, d_prime = np.nan, np.nan
        else:
            mask = np.asarray(mask)
            gcnr, d_prime = calculate_clinical_metrics(real, synth, mask)

        results[name]['gcnr'].append(gcnr)
        results[name]['d_prime'].append(d_prime)

# ==========================================
# STEP 4: ground-truth reference gCNR/d' (using real_flair as both "real" and "gen" is meaningless --
# instead compute GT detectability directly from real_flair vs itself's own fg/bg stats)
# ==========================================
def ground_truth_clinical_metrics(real_img, mask_img):
    fg_mask = mask_img > 0
    bg_mask = mask_img == 0
    if np.sum(fg_mask) == 0 or np.sum(bg_mask) == 0:
        return np.nan, np.nan
    fg = real_img[fg_mask].flatten()
    bg = real_img[bg_mask].flatten()
    mu_fg, mu_bg = np.mean(fg), np.mean(bg)
    std_fg, std_bg = np.std(fg), np.std(bg)
    pooled_noise = np.sqrt(0.5 * (std_fg**2 + std_bg**2))
    if pooled_noise == 0:
        return np.nan, np.nan
    signal_diff = np.abs(mu_fg - mu_bg)
    d_prime = signal_diff / pooled_noise
    z_score = signal_diff / np.sqrt(std_fg**2 + std_bg**2)
    gcnr = 1 - 2 * norm.cdf(-z_score / 2)
    return gcnr, d_prime

gt_gcnr_list, gt_dprime_list = [], []
any_folder = next(iter(METHOD_FOLDERS.values()))
for key in common_keys:
    d = torch.load(any_folder / f"{key}.pt", weights_only=False)
    mask = d.get('mask', None)
    if mask is not None:
        gcnr, dprime = ground_truth_clinical_metrics(np.asarray(d['real_flair']), np.asarray(mask))
        gt_gcnr_list.append(gcnr)
        gt_dprime_list.append(dprime)

gt_gcnr_arr = np.array(gt_gcnr_list, dtype=float)
gt_dprime_arr = np.array(gt_dprime_list, dtype=float)
gt_gcnr_mean = np.nanmean(gt_gcnr_arr)
gt_dprime_mean = np.nanmean(gt_dprime_arr)

# ==========================================
# STEP 5: build the summary table
# ==========================================
summary_rows = []
for name in METHOD_FOLDERS:
    row = {"Method": name}
    for metric_key, label in [('psnr', 'PSNR'), ('ssim', 'SSIM'), ('lpips', 'LPIPS'), ('mae', 'MAE')]:
        vals = np.array(results[name][metric_key], dtype=float)
        vals = vals[~np.isnan(vals)]
        row[label] = f"{np.mean(vals):.4f} ± {np.std(vals):.4f}" if len(vals) > 0 else "N/A"

    gcnr_vals = np.array(results[name]['gcnr'], dtype=float)
    gcnr_vals = gcnr_vals[~np.isnan(gcnr_vals)]
    dprime_vals = np.array(results[name]['d_prime'], dtype=float)
    dprime_vals = dprime_vals[~np.isnan(dprime_vals)]

    if len(gcnr_vals) > 0:
        gcnr_mean = np.mean(gcnr_vals)
        row['gCNR'] = f"{gcnr_mean:.4f} ± {np.std(gcnr_vals):.4f}"
        row['gCNR retention'] = f"{100 * gcnr_mean / gt_gcnr_mean:.1f}%"
    else:
        row['gCNR'], row['gCNR retention'] = "N/A", "N/A"

    if len(dprime_vals) > 0:
        dprime_mean = np.mean(dprime_vals)
        row["d'/gSNR"] = f"{dprime_mean:.4f} ± {np.std(dprime_vals):.4f}"
        row["gSNR retention"] = f"{100 * dprime_mean / gt_dprime_mean:.1f}%"
    else:
        row["d'/gSNR"], row["gSNR retention"] = "N/A", "N/A"

    row["n (all)"] = len(common_keys)
    row["n (tumor slices)"] = len(gcnr_vals)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("\n" + "=" * 100)
print(f"FAIR COMPARISON TABLE (n={len(common_keys)} total, ground-truth gCNR={gt_gcnr_mean:.4f}, gSNR={gt_dprime_mean:.4f})")
print(f"Using ONLY slice {SLICE_NUMBER} for all patients")
print("=" * 100)
print(summary_df.to_string(index=False))

# Also save the results with slice information in filename
summary_df.to_csv(f"fair_comparison_table_slice_{SLICE_NUMBER}.csv", index=False)
print(f"\nSaved to fair_comparison_table_slice_{SLICE_NUMBER}.csv")